# Binary multimodal lesion classification — isolated study v1
Train new binary heads on **MILK10k + ISIC 2019** using read-only saved
EfficientNetV2 encoders. **PAD-UFES-20 is external testing only**, opened after
models, calibration and decision thresholds are locked.

Attach (1) the new `skin_cancer_binary_v1.zip` code dataset, (2) MILK10k training
images, (3) official ISIC 2019 **training images, ground truth and metadata**,
and (4) PAD-UFES-20 **images and metadata**. Enable a GPU. T4 x2 runs one
extraction process per image view; one GPU works sequentially.

This is a new experiment, not a claim of improvement. The old eleven-class
baseline, report and Streamlit demo are not overwritten. Binary labels include
source-labelled malignancies and SCC in situ; actinic keratosis and
keratoacanthoma are excluded from the primary endpoint. Reference diagnoses
are labels only and never predictor inputs. Do not present new subset results
as the old full-cohort screening results.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, zipfile, stat, subprocess, hashlib, importlib.util
INPUT_ROOT = Path('/kaggle/input')
WORK = Path('/kaggle/working/skin-cancer-binary-v1')
PROJECT = WORK / 'project'
RUN = WORK / 'experiments/binary-v1'
ARTIFACT_ZIP = Path('/kaggle/working/binary_study_artifacts.zip')
CODE_INPUT = None        # Code ZIP or its extracted folder; auto-detects exactly one.
MILK_IMAGES = None       # .../MILK10k_Training_Input
ISIC_GROUND_TRUTH = None # .../ISIC_2019_Training_GroundTruth.csv
ISIC_METADATA = None     # .../ISIC_2019_Training_Metadata.csv
ISIC_IMAGES = None       # Folder containing extracted ISIC JPGs, recursively.
PAD_METADATA = None      # Official metadata.csv (patient_id, lesion_id, img_id, diagnostic).
PAD_IMAGES = None        # Folder containing extracted PAD PNGs, recursively.
RESUME_INPUT = None      # Previous binary_study_artifacts.zip; same inputs/config only.
SEEDS = [42, 2026, 3407]  # Three head-training seeds. Linear baseline fits once.
EPOCHS, PATIENCE, HEAD_BATCH_SIZE = 40, 7, 256
PREPROCESS_WORKERS = 2
PCA_DIMENSION = 128
ISIC_TRAIN_CAP = 0       # 0 uses all eligible ISIC training cases; cap affects TRAIN only.
CONDITIONS = ['clean', 'blur', 'jpeg', 'dark']
FEATURE_PRECISION = 'auto' # Tries FP32 and AMP; set 'fp32' for a new run if needed.
TARGET_SENSITIVITY = 0.95 # Calibration target, not a test-performance guarantee.
BOOTSTRAP_DRAWS = 1000
RUN_CHECKS = True
WORK.mkdir(parents=True, exist_ok=True)
print('Binary study setup is responding.', flush=True)


## Load and verify the new code bundle
Files are copied to a new writable project; input datasets stay read-only.
No nested source archive is required. Resume restores only this study's artifacts.


In [ ]:
import time, threading
from collections import defaultdict, deque

def index_dataset_roots(root, maximum_depth=5):
    files, directories = defaultdict(list), defaultdict(list)
    skip = {'milk10k_training_input', 'isic_2019_training_input', 'pad_ufes_20_input',
            'imgs_part_1', 'imgs_part_2', 'imgs_part_3', '__macosx', '.git', '__pycache__'}
    queue = deque([(Path(root), 0)])
    while queue:
        folder, depth = queue.popleft()
        STARTUP_STATUS['stage'] = 'Listing dataset root: ' + str(folder)
        print(STARTUP_STATUS['stage'], flush=True)
        # Never recurse into the large, known image directories.
        with os.scandir(folder) as entries:
            entries = [entry for entry in entries
                       if not entry.name.lower().endswith(('.jpg','.jpeg','.png','.bmp'))]
        children = []
        for entry in entries:
            path = Path(entry.path)
            if entry.is_dir():
                directories[entry.name].append(path)
                if depth < maximum_depth and entry.name.lower() not in skip:
                    children.append((path, depth + 1))
            elif entry.is_file():
                files[entry.name].append(path)
        if not any(entry.name == 'binary_bundle_manifest.json' for entry in entries):
            queue.extend(children)
    return dict(files), dict(directories)

STARTUP_STATUS = {'stage':'Beginning dataset discovery', 'started':time.monotonic()}
def startup_heartbeat(stop):
    while not stop.wait(15):
        print(f"Startup heartbeat: {STARTUP_STATUS['stage']} ({time.monotonic()-STARTUP_STATUS['started']:.0f}s elapsed)", flush=True)

def file_sha(path):
    with Path(path).open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()
def safe_unzip(path, target):
    target = Path(target).resolve(); target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(path) as archive:
        for member in archive.infolist():
            dest = (target / member.filename).resolve()
            if not dest.is_relative_to(target) or stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError('Unsafe ZIP member: ' + member.filename)
        for member in archive.infolist():
            STARTUP_STATUS['stage'] = 'Extracting code: ' + member.filename
            print(STARTUP_STATUS['stage'], flush=True)
            archive.extract(member, target)

stop = threading.Event()
ticker = threading.Thread(target=startup_heartbeat, args=(stop,), daemon=True)
ticker.start()
try:
    INPUT_FILE_INDEX, INPUT_DIRECTORY_INDEX = index_dataset_roots(INPUT_ROOT)
    if CODE_INPUT is None:
        archives = INPUT_FILE_INDEX.get('skin_cancer_binary_v1.zip', [])
        if len(archives) == 1:
            CODE_INPUT = archives[0]
        else:
            manifests = INPUT_FILE_INDEX.get('binary_bundle_manifest.json', [])
            if len(manifests) != 1:
                raise FileNotFoundError('Attach the NEW binary code dataset, or set CODE_INPUT explicitly. Discovery checks dataset roots up to five levels and skips image folders.')
            CODE_INPUT = manifests[0].parent
    CODE_INPUT = Path(CODE_INPUT)
    print('Located code input:', CODE_INPUT, flush=True)
    if CODE_INPUT.is_file():
        safe_unzip(CODE_INPUT, PROJECT)
    else:
        manifest = json.loads((CODE_INPUT / 'binary_bundle_manifest.json').read_text())
        PROJECT.mkdir(parents=True, exist_ok=True)
        for name in [*manifest['sha256'], 'binary_bundle_manifest.json']:
            src = (CODE_INPUT / name).resolve(); dest = (PROJECT / name).resolve()
            if not src.is_relative_to(CODE_INPUT.resolve()) or not dest.is_relative_to(PROJECT.resolve()):
                raise ValueError('Unsafe code path')
            STARTUP_STATUS['stage'] = 'Copying code: ' + name
            print(STARTUP_STATUS['stage'], flush=True)
            dest.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dest)
    manifest = json.loads((PROJECT / 'binary_bundle_manifest.json').read_text())
    for name, expected in manifest['sha256'].items():
        STARTUP_STATUS['stage'] = 'Verifying code: ' + name
        print(STARTUP_STATUS['stage'], flush=True)
        if file_sha(PROJECT / name) != expected:
            raise ValueError('Binary bundle failed integrity check: ' + name)
    REFERENCE = PROJECT / 'reference'
    if RESUME_INPUT is not None and not RUN.exists():
        STARTUP_STATUS['stage'] = 'Restoring previous binary study artifacts'
        safe_unzip(RESUME_INPUT, RUN)
    print('Verified new project:', PROJECT, flush=True)
    print('Separate output:', RUN, flush=True)
finally:
    stop.set(); ticker.join(timeout=1)

# ISIC_METADATA_REPAIR_V2: patch only the verified writable code copy.
METADATA_REPAIR_PAYLOAD = json.loads('{"src/binary_study/data.py": {"original_sha256": "7995b8848815cff6c0b7c95d1336dd001ad18fdd4e0180bfb44f4c9e67b9c2d6", "source": "\\"\\"\\"Source-specific labels, lesion aggregation and train-only shared metadata.\\"\\"\\"\\nfrom __future__ import annotations\\nimport json\\nfrom pathlib import Path\\nimport re\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.model_selection import train_test_split\\nfrom .common import write_json\\n\\nCLASSES = (\'AKIEC\',\'BCC\',\'BEN_OTH\',\'BKL\',\'DF\',\'INF\',\'MAL_OTH\',\'MEL\',\'NV\',\'SCCKA\',\'VASC\')\\nLABEL_POLICY = \'malignancy-including-scc-in-situ-v1; AK-and-KA-excluded\'\\nSUBTYPES = (\'BCC\',\'MEL\',\'SCC\',\'OTHER_MALIGNANT\')\\nSITES = (\'head_neck\',\'trunk\',\'upper_limb\',\'lower_limb\',\'acral\',\'genital\',\'other\',\'unknown\')\\nUNKNOWN = {\'\',\'nan\',\'none\',\'unk\',\'unknown\',\'na\',\'n/a\'}\\nISIC_METADATA_POLICY = \'isic-lesion-metadata-v2: valid-age-median; categorical-conflict-group-quarantine\'\\n\\n\\ndef milk_label(target, granular):\\n    target = int(target); text = str(granular).strip()\\n    if target not in range(11) or not text or text.lower() in UNKNOWN:\\n        raise ValueError(\'MILK reference diagnosis is missing/invalid\')\\n    if target == 0:\\n        if text == \'Solar or actinic keratosis\': return -1, -1\\n        if text == \'Squamous cell carcinoma in situ, Bowens disease\': return 1, 2\\n        raise ValueError(f\'Unreviewed AKIEC diagnosis: {text}\')\\n    if target == 9:\\n        if text == \'Keratoacanthoma\': return -1, -1\\n        if text == \'Squamous cell carcinoma, Invasive\': return 1, 2\\n        raise ValueError(f\'Unreviewed SCCKA diagnosis: {text}\')\\n    return (1, {1:0,7:1,6:3}[target]) if target in (1,6,7) else (0,-1)\\n\\n\\ndef source_label(code, source):\\n    mapping = {\'ISIC2019\': {\'BCC\':(1,0),\'MEL\':(1,1),\'SCC\':(1,2),\'AK\':(-1,-1),\\n                          \'NV\':(0,-1),\'BKL\':(0,-1),\'DF\':(0,-1),\'VASC\':(0,-1),\'UNK\':(-1,-1)},\\n               \'PAD\': {\'BCC\':(1,0),\'MEL\':(1,1),\'SCC\':(1,2),\'ACK\':(-1,-1),\\n                       \'NEV\':(0,-1),\'SEK\':(0,-1)}}\\n    if source not in mapping or code not in mapping[source]:\\n        raise ValueError(f\'Unreviewed {source} diagnosis: {code}\')\\n    return mapping[source][code]\\n\\n\\ndef safe_id(value):\\n    value = str(value)\\n    if not re.fullmatch(r\'[A-Za-z0-9_-]+\', value): raise ValueError(f\'Invalid identifier: {value}\')\\n    return value\\n\\n\\nclass ImageIndex:\\n    def __init__(self, root):\\n        self.root = Path(root).resolve(); self.paths = {}\\n        print(f\'Indexing source images: {self.root}\', flush=True)\\n        for p in self.root.rglob(\'*\'):\\n            if p.is_file() and p.suffix.lower() in (\'.jpg\',\'.jpeg\',\'.png\',\'.bmp\'):\\n                if not p.resolve().is_relative_to(self.root): raise ValueError(\'Image escapes dataset root\')\\n                key = p.stem\\n                if key in self.paths: raise ValueError(f\'Ambiguous image basename: {key}\')\\n                self.paths[key] = str(p.resolve())\\n                if len(self.paths)%5000 == 0:\\n                    print(f\'Indexed {len(self.paths)} source images\', flush=True)\\n        if not self.paths: raise FileNotFoundError(f\'No extracted images under {root}\')\\n        print(f\'Image index ready: {len(self.paths)} images\', flush=True)\\n\\n    def get(self, name):\\n        if Path(str(name)).name != str(name):raise ValueError(\'Image identifiers must be filenames, not paths\')\\n        key = safe_id(Path(str(name)).stem)\\n        if key not in self.paths: raise FileNotFoundError(f\'Image {key} not found in {self.root}\')\\n        return self.paths[key]\\n\\n\\ndef value(group, name):\\n    if name not in group: return None\\n    known = group[name].dropna().astype(str).str.strip()\\n    known = known[~known.str.lower().isin(UNKNOWN)].unique()\\n    if len(known) > 1: raise ValueError(f\'Conflicting lesion metadata: {name}\')\\n    return known[0] if len(known) else None\\n\\n\\ndef isic_metadata(group):\\n    \\"\\"\\"Aggregate covariates without labels; quarantine ambiguous lesion identities.\\n\\n    Repeated images may report different rounded ages. The median of valid\\n    observations is deterministic. Conflicting sex or coarse site may indicate\\n    a reused lesion identifier, so the whole group is excluded downstream.\\n    Raw observations are retained for audit; image/diagnosis checks stay strict.\\n    \\"\\"\\"\\n    ages=pd.to_numeric(group.age_approx,errors=\'coerce\')\\n    valid=ages[ages.between(0,120)]\\n    age=float(valid.median()) if len(valid) else None\\n    unique_ages=sorted(float(x) for x in valid.unique())\\n    def known(name):\\n        return sorted({str(x).strip().lower() for x in group[name].dropna()\\n                       if str(x).strip().lower() not in UNKNOWN})\\n    sexes=known(\'sex\');raw_sites=known(\'anatom_site_general\')\\n    sites=sorted({site_code(x) for x in raw_sites}-{ \'unknown\' })\\n    reasons=[]\\n    if len(sexes)>1:reasons.append(\'conflicting_sex\')\\n    if set(sexes)-{\'male\',\'female\'}:reasons.append(\'unrecognized_sex\')\\n    if len(sites)>1:reasons.append(\'conflicting_coarse_site\')\\n    return age,(sexes[0] if len(sexes)==1 else None),(sites[0] if len(sites)==1 else None),{\\n        \'metadata_policy\':ISIC_METADATA_POLICY,\\n        \'metadata_quarantine\':bool(reasons),\\n        \'metadata_quarantine_reasons\':json.dumps(reasons),\\n        \'metadata_age_varied\':len(unique_ages)>1,\\n        \'metadata_observations\':json.dumps({\'ages\':unique_ages,\'valid_age_rows\':len(valid),\\n            \'missing_or_invalid_age_rows\':int(len(ages)-len(valid)),\\n            \'sex\':sexes,\'raw_sites\':raw_sites,\'coarse_sites\':sites},sort_keys=True)}\\n\\n\\ndef _case(source, lesion, group, split, clinical, dermoscopic, target, subtype, age, sex, site, **extra):\\n    return {\'case_id\':f\'{source}:{safe_id(lesion)}\',\'source\':source,\'group_id\':f\'{source}:{safe_id(group)}\',\\n            \'split\':split,\'clinical\':json.dumps(clinical),\'dermoscopic\':json.dumps(dermoscopic),\\n            \'binary\':int(target),\'subtype\':int(subtype),\'age\':age,\'sex\':sex,\'site\':site,**extra}\\n\\n\\ndef milk_cases(csv, images):\\n    frame = pd.read_csv(csv, dtype={\'lesion_id\':str,\'isic_id\':str})\\n    required = {\'lesion_id\',\'isic_id\',\'image_type\',\'target\',\'split\',\'diagnosis_full\',\'age_approx\',\'sex\',\'site\'}\\n    if not required <= set(frame): raise ValueError(f\'MILK CSV missing {required-set(frame)}\')\\n    if frame.isic_id.duplicated().any(): raise ValueError(\'Duplicate MILK image ID\')\\n    index = ImageIndex(images); rows = []\\n    patients = next((c for c in (\'patient_id\',\'case_id\',\'subject_id\',\'patient\') if c in frame),None)\\n    for lesion,g in frame.groupby(\'lesion_id\',sort=True):\\n        if len(g)!=2 or set(g.image_type)!= {\'clinical: close-up\',\'dermoscopic\'}:\\n            raise ValueError(f\'MILK pair incomplete: {lesion}\')\\n        for c in (\'split\',\'target\',\'diagnosis_full\',\'age_approx\',\'sex\',\'site\'):\\n            if g[c].nunique(dropna=False)!=1: raise ValueError(f\'Conflicting MILK pair {lesion}: {c}\')\\n        if g[\'split\'].iloc[0] not in (\'train\',\'val\',\'fusion\',\'holdout\'): raise ValueError(\'Invalid frozen split\')\\n        target,subtype=milk_label(g.target.iloc[0],g.diagnosis_full.iloc[0])\\n        patient=value(g,patients) if patients else None\\n        rows.append(_case(\'MILK\',lesion,patient or lesion,g[\'split\'].iloc[0],\\n          [index.get(g.loc[g.image_type==\'clinical: close-up\',\'isic_id\'].iloc[0])],\\n          [index.get(g.loc[g.image_type==\'dermoscopic\',\'isic_id\'].iloc[0])],target,subtype,\\n          value(g,\'age_approx\'),value(g,\'sex\'),value(g,\'site\'),original_target=int(g.target.iloc[0]),\\n          patient_grouping=bool(patient),diagnosis_confirm_type=value(g,\'diagnosis_confirm_type\')))\\n    result=pd.DataFrame(rows); assert_groups(result)\\n    return result\\n\\n\\ndef isic_cases(ground_truth, metadata, images, seed=42):\\n    gt=pd.read_csv(ground_truth,dtype={\'image\':str});meta=pd.read_csv(metadata,dtype={\'image\':str})\\n    labels=[\'MEL\',\'NV\',\'BCC\',\'AK\',\'BKL\',\'DF\',\'VASC\',\'SCC\']\\n    metadata_fields={\'image\',\'lesion_id\',\'age_approx\',\'sex\',\'anatom_site_general\'}\\n    if not {\'image\',*labels} <= set(gt) or not metadata_fields <= set(meta):\\n        raise ValueError(\'Use official ISIC 2019 ground truth and metadata, including lesion_id, age_approx, sex and anatom_site_general\')\\n    cols=labels+([\'UNK\'] if \'UNK\' in gt else [])\\n    a=gt[cols].to_numpy(float)\\n    if not np.isfinite(a).all() or not np.isin(a,[0,1]).all() or not (a.sum(1)==1).all():\\n        raise ValueError(\'ISIC ground truth must be one-hot diagnoses\')\\n    gt=gt.assign(diagnosis=[cols[i] for i in a.argmax(1)])\\n    if gt.image.duplicated().any() or meta.image.duplicated().any(): raise ValueError(\'Repeated ISIC image identity\')\\n    merged=gt.merge(meta,on=\'image\',how=\'left\',validate=\'one_to_one\',indicator=True)\\n    if not merged[\'_merge\'].eq(\'both\').all(): raise ValueError(\'ISIC metadata rows missing\')\\n    index=ImageIndex(images); merged[\'lesion\']=merged.get(\'lesion_id\',merged.image).fillna(merged.image).astype(str)\\n    rows=[]\\n    for lesion,g in merged.groupby(\'lesion\',sort=True):\\n        if g.diagnosis.nunique()!=1: raise ValueError(\'ISIC lesion has conflicting labels\')\\n        target,subtype=source_label(g.diagnosis.iloc[0],\'ISIC2019\')\\n        patient=value(g,\'patient_id\')\\n        age,sex,site,metadata_audit=isic_metadata(g)\\n        rows.append(_case(\'ISIC2019\',lesion,patient or lesion,\'unassigned\',[],[index.get(x) for x in g.image],\\n          target,subtype,age,sex,site,\\n          original_target=-1,patient_grouping=bool(patient),diagnosis_confirm_type=None,**metadata_audit))\\n    result=pd.DataFrame(rows)\\n    # Groups, not images, are assigned. All images of one lesion remain one case.\\n    groups=result.groupby(\'group_id\').binary.max().clip(lower=0)\\n    if groups.value_counts().min()<4: raise ValueError(\'Insufficient groups for stratified ISIC development splits\')\\n    fit,cal=train_test_split(groups.index,test_size=.15,random_state=seed,stratify=groups)\\n    train,val=train_test_split(fit,test_size=.15/.85,random_state=seed+1,stratify=groups.loc[fit])\\n    assignments={**{x:\'train\' for x in train},**{x:\'val\' for x in val},**{x:\'fusion\' for x in cal}}\\n    result[\'split\']=result.group_id.map(assignments); assert_groups(result)\\n    print(f\'ISIC metadata: {len(result)} lesions; {int(result.metadata_age_varied.sum())} with varying ages; \'\\n          f\'{int(result.metadata_quarantine.sum())} categorical-conflict cases marked for whole-group quarantine\',flush=True)\\n    return result\\n\\n\\ndef pad_cases(metadata, images):\\n    frame=pd.read_csv(metadata,dtype={\'patient_id\':str,\'lesion_id\':str,\'img_id\':str})\\n    required={\'patient_id\',\'lesion_id\',\'img_id\',\'diagnostic\',\'age\',\'gender\',\'region\'}\\n    if not required<=set(frame): raise ValueError(\'PAD metadata must have official patient, lesion, image and diagnosis fields\')\\n    if frame[list(required)].isna().any().any() and frame[[\'patient_id\',\'lesion_id\',\'img_id\',\'diagnostic\']].isna().any().any():\\n        raise ValueError(\'PAD identities and diagnoses cannot be missing\')\\n    if frame.img_id.duplicated().any(): raise ValueError(\'Repeated PAD image identity\')\\n    index=ImageIndex(images); rows=[]\\n    for (patient,lesion),g in frame.groupby([\'patient_id\',\'lesion_id\'],sort=True):\\n        if g.diagnostic.nunique()!=1: raise ValueError(\'PAD lesion labels disagree\')\\n        target,subtype=source_label(g.diagnostic.iloc[0],\'PAD\')\\n        rows.append(_case(\'PAD\',f\'{safe_id(patient)}_{safe_id(lesion)}\',patient,\'external\',\\n          [index.get(x) for x in g.img_id],[],target,subtype,value(g,\'age\'),value(g,\'gender\'),value(g,\'region\'),\\n          original_target=-1,patient_grouping=True,diagnosis_confirm_type={\'true\':\'histopathology\',\'1\':\'histopathology\',\'false\':\'clinical\',\'0\':\'clinical\'}.get(str(value(g,\'biopsed\')).lower(),\'unknown\')))\\n    result=pd.DataFrame(rows);assert_groups(result)\\n    return result\\n\\n\\ndef assert_groups(frame):\\n    if frame.case_id.duplicated().any(): raise ValueError(\'Duplicate case IDs\')\\n    if (frame.groupby(\'group_id\')[\'split\'].nunique()>1).any(): raise ValueError(\'Patient/lesion group crosses splits\')\\n    if not frame.binary.isin([-1,0,1]).all(): raise ValueError(\'Invalid binary target\')\\n    owners={}\\n    for row in frame.itertuples():\\n        for view in (\'clinical\',\'dermoscopic\'):\\n            for path in json.loads(getattr(row,view)):\\n                if path in owners and owners[path]!=row.case_id: raise ValueError(\'Image assigned to multiple lesions\')\\n                owners[path]=row.case_id\\n\\n\\ndef site_code(raw):\\n    s=str(raw).lower().strip().replace(\'_\',\' \')\\n    if s in UNKNOWN:return \'unknown\'\\n    if s in (\'acral\',\'palm\',\'palms\',\'soles\',\'sole\'):return \'acral\'\\n    if any(t in s for t in (\'head\',\'neck\',\'face\',\'scalp\',\'ear\',\'nose\',\'lip\')):return \'head_neck\'\\n    if any(t in s for t in (\'arm\',\'upper\',\'hand\',\'forearm\',\'wrist\')):return \'upper_limb\'\\n    if any(t in s for t in (\'leg\',\'lower\',\'thigh\',\'foot\',\'feet\',\'ankle\')):return \'lower_limb\'\\n    if any(t in s for t in (\'trunk\',\'torso\',\'chest\',\'back\',\'abdomen\',\'breast\',\'buttock\')):return \'trunk\'\\n    if any(t in s for t in (\'genital\',\'groin\')):return \'genital\'\\n    return \'other\'\\n\\n\\nclass SharedMetadata:\\n    \\"\\"\\"Age, sex, coarse site and per-field observed flags; no labels or source input.\\"\\"\\"\\n    width=1+3+len(SITES)+3\\n    def fit(self, frame):\\n        if not set(frame[\'split\'])<= {\'train\'}: raise ValueError(\'Metadata may fit training cases only\')\\n        ages=pd.to_numeric(frame.age,errors=\'coerce\');ages=ages.where(ages.between(0,120))\\n        self.mean=float(ages.mean()) if ages.notna().any() else 60.\\n        self.std=max(float(ages.std(ddof=0)) if ages.notna().any() else 1.,1.)\\n        return self\\n    def transform(self, frame):\\n        x=np.zeros((len(frame),self.width),np.float32); observed=np.zeros((len(frame),3),np.float32)\\n        for i,r in enumerate(frame.itertuples()):\\n            age=pd.to_numeric(getattr(r,\'age\',None),errors=\'coerce\')\\n            known=bool(np.isfinite(age) and 0<=age<=120)\\n            x[i,0]=(float(age)-self.mean)/self.std if known else 0.; observed[i,0]=known\\n            sex=str(getattr(r,\'sex\',None)).strip().lower();sex={\'male\':0,\'female\':1}.get(sex,2)\\n            x[i,1+sex]=1.;observed[i,1]=sex!=2\\n            site=site_code(getattr(r,\'site\',None));x[i,4+SITES.index(site)]=1.;observed[i,2]=site!=\'unknown\'\\n        x[:,-3:]=observed\\n        return x,observed\\n    def save(self,path):write_json(path,{\'mean\':self.mean,\'std\':self.std,\'width\':self.width,\'fields\':[\'age\',\'sex\',\'site\']})\\n    @classmethod\\n    def load(cls,path):\\n        data=json.loads(Path(path).read_text());obj=cls();obj.mean=data[\'mean\'];obj.std=data[\'std\'];return obj\\n\\n\\ndef mask_metadata(values, flags):\\n    x=values.copy(); x[:,0]*=flags[:,0];x[:,1:4]*=flags[:,1,None];x[:,4:4+len(SITES)]*=flags[:,2,None];x[:,-3:]=flags\\n    return x\\n", "sha256": "5ad3bb00c654a47ce2ce359b0ee20ee90340df120257bbc18c48a0ca62d30ca8"}, "src/binary_study/overlap.py": {"original_sha256": "85f65a257da752b00857ae8f8171bd7664693afcdd0b0047f5bbfc55d878a894", "source": "\\"\\"\\"Pixel/identity overlap quarantine. Similarity is conservative, not proof of duplication.\\"\\"\\"\\nfrom __future__ import annotations\\nfrom concurrent.futures import ThreadPoolExecutor\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\nimport time\\nimport numpy as np\\nimport pandas as pd\\nfrom PIL import Image,ImageOps\\nfrom scipy.fft import dctn\\nfrom .common import sha,write_json,digest\\nfrom .data import ISIC_METADATA_POLICY\\n\\n\\ndef image_records(frame):\\n    rows=[]\\n    for r in frame.itertuples():\\n        for view in (\'clinical\',\'dermoscopic\'):\\n            for path in json.loads(getattr(r,view)):\\n                key=digest([r.source,r.case_id,view,Path(path).stem])[:32]\\n                rows.append({\'key\':key,\'case_id\':r.case_id,\'group_id\':r.group_id,\'source\':r.source,\\n                             \'split\':r.split,\'view\':view,\'path\':path,\'image_id\':Path(path).stem})\\n    return rows\\n\\n\\ndef fingerprint(record):\\n    path=Path(record[\'path\']);stat=path.stat()\\n    with Image.open(path) as im:\\n        rgb=ImageOps.exif_transpose(im).convert(\'RGB\')\\n        pixel=hashlib.sha256(str(rgb.size).encode()+rgb.tobytes()).hexdigest()\\n        base=rgb.resize((32,32),Image.Resampling.LANCZOS).convert(\'L\');hashes=[]\\n        for reflected in [False,True]:\\n            image=base.transpose(Image.Transpose.FLIP_LEFT_RIGHT) if reflected else base\\n            for rotation in (0,90,180,270):\\n                low=dctn(np.asarray(image.rotate(rotation),dtype=float),norm=\'ortho\')[:8,:8].flatten()\\n                bits=low>np.median(low[1:]);hashes.append(int.from_bytes(np.packbits(bits).tobytes(),\'big\'))\\n    return {**record,\'sha256\':sha(path),\'pixel_sha256\':pixel,\'phash\':[str(x) for x in hashes],\\n            \'size\':stat.st_size,\'mtime_ns\':stat.st_mtime_ns}\\n\\n\\ndef fingerprints(frame, cache, workers=2):\\n    records=image_records(frame);cache=Path(cache);prior={}\\n    if cache.exists():prior={r[\'key\']:r for r in json.loads(cache.read_text())}\\n    def one(r):\\n        old=prior.get(r[\'key\']);s=Path(r[\'path\']).stat()\\n        if old and old[\'path\']==r[\'path\'] and old[\'size\']==s.st_size and old[\'mtime_ns\']==s.st_mtime_ns:\\n            return {**old,**r}\\n        return fingerprint(r)\\n    result=[];start=time.monotonic()\\n    with ThreadPoolExecutor(max_workers=max(1,workers)) as pool:\\n        for i,r in enumerate(pool.map(one,records),1):\\n            result.append(r)\\n            if i%250==0 or i==len(records):\\n                print(f\'Overlap audit {i}/{len(records)} images ({time.monotonic()-start:.1f}s)\',flush=True)\\n                if i%2000==0:write_json(cache,result)\\n    write_json(cache,result);return result\\n\\n\\nclass FingerprintIndex:\\n    \\"\\"\\"Pigeonhole index: <=radius bit changes imply one unchanged segment.\\"\\"\\"\\n    def __init__(self, radius=3):\\n        if not 0<=radius<=8:raise ValueError(\'pHash radius must be 0..8\')\\n        self.radius=radius;self.exact={};self.ids={};self.buckets={};self.entries=[]\\n        self.parts=radius+1\\n    def segments(self,value):\\n        for i in range(self.parts):\\n            lo=64*i//self.parts;hi=64*(i+1)//self.parts\\n            yield i,(value>>lo)&((1<<(hi-lo))-1)\\n    def add(self,r):\\n        index=len(self.entries);self.entries.append(r)\\n        self.exact.setdefault(r[\'sha256\'],set()).add(index)\\n        self.exact.setdefault(r[\'pixel_sha256\'],set()).add(index)\\n        self.ids.setdefault(r[\'image_id\'],set()).add(index)\\n        for v in set(map(int,r[\'phash\'])):\\n            for segment in self.segments(v):self.buckets.setdefault(segment,set()).add((index,v))\\n    def matches(self,r):\\n        found={i:\'exact_file_or_pixels\' for h in [r[\'sha256\'],r[\'pixel_sha256\']] for i in self.exact.get(h,())}\\n        found.update({i:\'shared_image_identifier\' for i in self.ids.get(r[\'image_id\'],())})\\n        v=int(r[\'phash\'][0]);candidates=set()\\n        for segment in self.segments(v):candidates.update(self.buckets.get(segment,()))\\n        for i,old in candidates:\\n            if (v^old).bit_count()<=self.radius:found.setdefault(i,\'conservative_near_duplicate\')\\n        return [(self.entries[i],why) for i,why in found.items()]\\n\\n\\ndef quarantine_development(frame, records, radius=3):\\n    index=FingerprintIndex(radius);reject=set();audit=[]\\n    metadata_cases=[]\\n    if \'metadata_quarantine\' in frame:\\n        flagged=frame[frame.metadata_quarantine.fillna(False).astype(bool)]\\n        reject.update(flagged.group_id)\\n        metadata_cases=[{\'case_id\':r.case_id,\'group_id\':r.group_id,\\n                         \'reasons\':json.loads(r.metadata_quarantine_reasons),\\n                         \'observations\':json.loads(r.metadata_observations)}\\n                        for r in flagged.itertuples()]\\n    # Audit every MILK partition, including diagnoses excluded from this new endpoint.\\n    for r in records:\\n        if r[\'source\']==\'MILK\':\\n            for previous,reason in index.matches(r):\\n                if (previous[\'group_id\']!=r[\'group_id\'] and previous[\'split\']!=r[\'split\']\\n                    and reason==\'exact_file_or_pixels\'):\\n                    reject.update([previous[\'group_id\'],r[\'group_id\']])\\n                    audit.append({\'case\':r[\'case_id\'],\'matched_case\':previous[\'case_id\'],\'reason\':reason,\'action\':\'Quarantine both groups in the NEW study; frozen split unchanged\'})\\n                elif previous[\'group_id\']!=r[\'group_id\'] and previous[\'split\']!=r[\'split\']:\\n                    reject.update([previous[\'group_id\'],r[\'group_id\']])\\n                    audit.append({\'case\':r[\'case_id\'],\'matched_case\':previous[\'case_id\'],\'reason\':reason,\'action\':\'Conservative cross-split MILK quarantine in new study only\'})\\n            index.add(r)\\n    for r in records:\\n        if r[\'source\']!=\'ISIC2019\':continue\\n        for previous,reason in index.matches(r):\\n            if previous[\'group_id\']==r[\'group_id\']:continue\\n            reject.add(r[\'group_id\'])\\n            if previous[\'source\']==\'ISIC2019\':reject.add(previous[\'group_id\'])\\n            audit.append({\'case\':r[\'case_id\'],\'matched_case\':previous[\'case_id\'],\'reason\':reason})\\n        index.add(r)\\n    result=frame[(frame.binary>=0)&~frame.group_id.isin(reject)].reset_index(drop=True)\\n    return result,{\'quarantined_groups\':sorted(reject),\'matches\':audit,\'radius\':radius,\\n                   \'metadata_policy\':ISIC_METADATA_POLICY,\\n                   \'metadata_conflict_cases\':metadata_cases,\\n                   \'metadata_conflict_groups\':sorted({r[\'group_id\'] for r in metadata_cases}),\\n                   \'isic_cases_with_varying_age\':int(frame.get(\'metadata_age_varied\',pd.Series(False,index=frame.index)).fillna(False).astype(bool).sum()),\\n                   \'excluded_diagnosis_cases\':int((frame.binary<0).sum()),\\n                   \'limitations\':\'Conservative pHash quarantine may exclude similar distinct lesions; cropped duplicates can remain.\'}\\n\\n\\ndef quarantine_external(frame, records, development_records, radius=3):\\n    index=FingerprintIndex(radius)\\n    for r in development_records:index.add(r)\\n    reject=set();audit=[]\\n    for r in records:\\n        for previous,reason in index.matches(r):\\n            if previous[\'source\']==\'PAD\' and previous[\'group_id\']==r[\'group_id\']:continue\\n            reject.add(r[\'group_id\'])\\n            if previous[\'source\']==\'PAD\':reject.add(previous[\'group_id\'])\\n            audit.append({\'case\':r[\'case_id\'],\'matched_case\':previous[\'case_id\'],\'reason\':reason})\\n        index.add(r)\\n    result=frame[(frame.binary>=0)&~frame.group_id.isin(reject)].reset_index(drop=True)\\n    return result,{\'quarantined_patient_groups\':sorted(reject),\'matches\':audit,\'radius\':radius,\\n                   \'excluded_diagnosis_cases\':int((frame.binary<0).sum()),\\n                   \'original_cases\':len(frame),\'eligible_test_cases\':len(result),\\n                   \'policy\':\'Whole external patient quarantined using pixels/IDs only, never model errors or labels.\'}\\n", "sha256": "0dbfd36807fce689d686a9c86d0eaabdc8e01e252652c20a670adae837d1dabd"}, "tests/test_binary_study_metadata.py": {"original_sha256": null, "source": "\\"\\"\\"Regression coverage for official ISIC repeated-image metadata conflicts.\\"\\"\\"\\nimport json\\nfrom pathlib import Path\\nimport sys\\nimport tempfile\\nimport unittest\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom PIL import Image\\n\\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]/\'src\'))\\nfrom binary_study.data import isic_cases,isic_metadata\\nfrom binary_study.overlap import image_records,quarantine_development\\n\\n\\nclass ISICMetadataTests(unittest.TestCase):\\n    def test_age_median_is_label_blind_order_independent_and_audited(self):\\n        frame=pd.DataFrame({\'age_approx\':[50,55,np.nan,999], \'sex\':[\'Male\',\'male\',None,\'unknown\'],\\n                            \'anatom_site_general\':[\'posterior torso\',\'anterior torso\',None,None],\\n                            \'diagnosis\':[\'NV\']*4,\'binary\':[0]*4})\\n        age,sex,site,audit=isic_metadata(frame)\\n        self.assertEqual((age,sex,site),(52.5,\'male\',\'trunk\'))\\n        self.assertFalse(audit[\'metadata_quarantine\']);self.assertTrue(audit[\'metadata_age_varied\'])\\n        self.assertEqual(json.loads(audit[\'metadata_observations\'])[\'missing_or_invalid_age_rows\'],2)\\n        self.assertEqual(isic_metadata(frame),isic_metadata(frame.iloc[::-1].assign(diagnosis=\'MEL\',binary=1)))\\n\\n    def test_sex_and_coarse_site_conflicts_are_flagged(self):\\n        frame=pd.DataFrame({\'age_approx\':[55,55],\'sex\':[\'male\',\'female\'],\\n                            \'anatom_site_general\':[\'head/neck\',\'posterior torso\']})\\n        _age,sex,site,audit=isic_metadata(frame)\\n        self.assertIsNone(sex);self.assertIsNone(site)\\n        self.assertEqual(json.loads(audit[\'metadata_quarantine_reasons\']),[\'conflicting_sex\',\'conflicting_coarse_site\'])\\n\\n    def test_unknown_fields_remain_missing(self):\\n        frame=pd.DataFrame({\'age_approx\':[\'unknown\',-1],\'sex\':[None,\'unknown\'],\\n                            \'anatom_site_general\':[\'unknown\',None]})\\n        age,sex,site,audit=isic_metadata(frame)\\n        self.assertEqual((age,sex,site),(None,None,None));self.assertFalse(audit[\'metadata_quarantine\'])\\n\\n    def fixture(self,root):\\n        images=root/\'images\';images.mkdir();rows=[]\\n        for i in range(30):\\n            diag=\'BCC\' if i%2 else \'NV\'\\n            for j in range(2):\\n                name=f\'ISIC_{i:04d}_{j}\'\\n                Image.new(\'RGB\',(8,8),(i,j,50)).save(images/f\'{name}.jpg\')\\n                rows.append({\'image\':name,**{d:int(d==diag) for d in [\'MEL\',\'NV\',\'BCC\',\'AK\',\'BKL\',\'DF\',\'VASC\',\'SCC\']},\\n                             \'lesion_id\':f\'lesion_{i}\',\'age_approx\':50+5*j if i==0 else 50,\\n                             \'sex\':\'female\' if i==1 and j==1 else \'male\',\\n                             \'anatom_site_general\':\'head/neck\' if i==2 and j==1 else \'posterior torso\'})\\n        frame=pd.DataFrame(rows);gt=root/\'gt.csv\';meta=root/\'meta.csv\'\\n        frame[[\'image\',\'MEL\',\'NV\',\'BCC\',\'AK\',\'BKL\',\'DF\',\'VASC\',\'SCC\']].to_csv(gt,index=False)\\n        frame[[\'image\',\'lesion_id\',\'age_approx\',\'sex\',\'anatom_site_general\']].to_csv(meta,index=False)\\n        return frame,gt,meta,images\\n\\n    def test_real_adapter_retains_images_and_quarantines_whole_group(self):\\n        with tempfile.TemporaryDirectory() as temp:\\n            frame,gt,meta,images=self.fixture(Path(temp));cases=isic_cases(gt,meta,images)\\n            self.assertEqual(len(cases),30);self.assertEqual(len(image_records(cases)),60)\\n            first=cases[cases.case_id.eq(\'ISIC2019:lesion_0\')].iloc[0]\\n            self.assertEqual(first.age,52.5)\\n            self.assertEqual(int(cases.metadata_quarantine.sum()),2)\\n            # Another case belonging to the same known patient is also quarantined.\\n            cases.loc[cases.case_id.eq(\'ISIC2019:lesion_3\'),\'group_id\']=\'ISIC2019:lesion_1\'\\n            retained,audit=quarantine_development(cases,[])\\n            self.assertEqual(len(retained),27)\\n            self.assertEqual(audit[\'metadata_conflict_groups\'],[\'ISIC2019:lesion_1\',\'ISIC2019:lesion_2\'])\\n            self.assertEqual(audit[\'isic_cases_with_varying_age\'],1)\\n            self.assertTrue(cases.binary.isin([0,1]).all())\\n            self.assertEqual(set(retained[\'split\']),{\'train\',\'val\',\'fusion\'})\\n\\n    def test_diagnosis_and_patient_conflicts_still_fail(self):\\n        with tempfile.TemporaryDirectory() as temp:\\n            frame,gt,meta,images=self.fixture(Path(temp))\\n            labels=pd.read_csv(gt);labels.loc[1,[\'NV\',\'BCC\']]=[0,1];labels.to_csv(gt,index=False)\\n            with self.assertRaisesRegex(ValueError,\'conflicting labels\'):isic_cases(gt,meta,images)\\n            frame[[\'image\',\'MEL\',\'NV\',\'BCC\',\'AK\',\'BKL\',\'DF\',\'VASC\',\'SCC\']].to_csv(gt,index=False)\\n            metadata=pd.read_csv(meta);metadata[\'patient_id\']=\'patient_1\';metadata.loc[1,\'patient_id\']=\'patient_2\';metadata.to_csv(meta,index=False)\\n            with self.assertRaisesRegex(ValueError,\'patient_id\'):isic_cases(gt,meta,images)\\n\\n\\nif __name__==\'__main__\':unittest.main()\\n", "sha256": "c4414f36b4613608412b0cf999b7151c2d4bf8a9a19d039fd2ca44cce50072fc"}}')

repair_receipt = {'format':'isic-metadata-repair-v2','files':{},
                  'policy':'Median of valid within-lesion ages; categorical identity conflicts quarantine the whole group before fitting.'}
reference_before = {n:file_sha(PROJECT/n) for n in manifest['sha256'] if n.startswith('reference/')}
for name, item in METADATA_REPAIR_PAYLOAD.items():
    destination = (PROJECT/name).resolve()
    if not destination.is_relative_to(PROJECT.resolve()) or not name.startswith(('src/binary_study/','tests/')):
        raise ValueError('Unexpected metadata repair destination')
    current = file_sha(destination) if destination.exists() else None
    if current not in (item['original_sha256'],item['sha256']):
        raise ValueError('Unrecognized source for pinned metadata repair: '+name)
    compile(item['source'],name,'exec')
    destination.parent.mkdir(parents=True,exist_ok=True)
    destination.write_text(item['source'])
    if file_sha(destination) != item['sha256']:
        raise ValueError('Metadata repair write failed: '+name)
    repair_receipt['files'][name] = {'original_sha256':item['original_sha256'],'repaired_sha256':item['sha256']}
if reference_before != {n:file_sha(PROJECT/n) for n in reference_before}:
    raise ValueError('Reference changed during metadata repair')
repair_receipt['unchanged_reference_sha256'] = reference_before
(PROJECT/'metadata_repair.json').write_text(json.dumps(repair_receipt,indent=2)+'\n')
print('ISIC metadata repair v2 installed and verified. Reference weights and frozen splits unchanged.',flush=True)


## Check dependencies and locate the three datasets
Keep the CUDA torch/torchvision versions provided by Kaggle. Input discovery
reads PAD **headers only**; external records and labels are not used here.
If a path is ambiguous, set its override above. ZIP image archives must already
be extracted by Kaggle; compressed image archives are not model inputs.


In [ ]:
required = {'numpy':'numpy', 'pandas':'pandas', 'scipy':'scipy', 'PIL':'Pillow',
            'cv2':'opencv-python-headless', 'matplotlib':'matplotlib', 'joblib':'joblib', 'sklearn':'scikit-learn>=1.4,<2'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
print('Loading PyTorch...', flush=True)
import torch, torchvision, pandas as pd
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU accelerator before starting this notebook.')
print('PyTorch', torch.__version__, 'torchvision', torchvision.__version__, flush=True)
for i in range(torch.cuda.device_count()):
    print('GPU', i, torch.cuda.get_device_name(i), flush=True)
def unique(pattern, label):
    paths = INPUT_FILE_INDEX.get(pattern, []) + INPUT_DIRECTORY_INDEX.get(pattern, [])
    if len(paths) != 1:
        raise FileNotFoundError(f'Set {label} explicitly: found {len(paths)} matches for {pattern}')
    return paths[0]
MILK_IMAGES = Path(MILK_IMAGES) if MILK_IMAGES else unique('MILK10k_Training_Input', 'MILK_IMAGES')
ISIC_GROUND_TRUTH = Path(ISIC_GROUND_TRUTH) if ISIC_GROUND_TRUTH else unique('ISIC_2019_Training_GroundTruth.csv', 'ISIC_GROUND_TRUTH')
ISIC_METADATA = Path(ISIC_METADATA) if ISIC_METADATA else unique('ISIC_2019_Training_Metadata.csv', 'ISIC_METADATA')
ISIC_IMAGES = Path(ISIC_IMAGES) if ISIC_IMAGES else ISIC_GROUND_TRUTH.parent
if PAD_METADATA is None:
    matches = []
    for path in [p for name, paths in INPUT_FILE_INDEX.items() if name.lower().endswith('.csv') for p in paths]:
        try:
            columns = set(pd.read_csv(path, nrows=0).columns)
        except Exception:
            continue
        if {'patient_id', 'lesion_id', 'img_id', 'diagnostic', 'gender', 'region'} <= columns:
            matches.append(path)
    if len(matches) != 1:
        raise FileNotFoundError('Attach official PAD metadata and set PAD_METADATA explicitly if ambiguous.')
    PAD_METADATA = matches[0]
PAD_METADATA = Path(PAD_METADATA)
PAD_IMAGES = Path(PAD_IMAGES) if PAD_IMAGES else PAD_METADATA.parent
for name in ['MILK_IMAGES','ISIC_GROUND_TRUTH','ISIC_METADATA','ISIC_IMAGES','PAD_METADATA','PAD_IMAGES']:
    print(name, '=', globals()[name], flush=True)
    if not globals()[name].exists():
        raise FileNotFoundError(str(globals()[name]))


In [ ]:
def run_cli(*arguments):
    command = [sys.executable, '-u', str(PROJECT/'src/scripts/binary_study.py'),
               *map(str, arguments), '--reference', str(REFERENCE), '--output', str(RUN)]
    print('Starting:', arguments[0], flush=True)
    process = subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
        if code:
            raise subprocess.CalledProcessError(code, command)
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=20)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
        raise
def stage(*arguments):
    try:
        run_cli(*arguments)
    except BaseException:
        if (RUN/'identity.json').exists():
            run_cli('package', '--destination', ARTIFACT_ZIP)
        raise
run_cli('smoke-gpu')
shutil.copy2(PROJECT/'metadata_repair.json', RUN/'metadata_repair.json')
if RUN_CHECKS:
    env = dict(os.environ, PYTHONPATH=str(PROJECT/'src'))
    test_log = RUN/'logs/focused_tests.log'
    test_log.parent.mkdir(parents=True,exist_ok=True)
    print('Running focused CPU regression checks; full log:',test_log,flush=True)
    with test_log.open('w') as handle:
        tests = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests',
                                '-p', 'test_binary_study*.py', '-v'], cwd=PROJECT, env=env,
                               stdout=handle,stderr=subprocess.STDOUT,timeout=300)
    print('\n'.join(test_log.read_text(errors='replace').splitlines()[-12:]),flush=True)
    tests.check_returncode()
print('Binary-specific CPU checks passed or explicitly skipped. No fork-based DataLoader tests.', flush=True)


## Prepare development data and audit overlaps
Uses MILK + ISIC only. Original MILK partitions stay unchanged. ISIC groups
are split 70/15/15 into train/validation/calibration. Near-duplicate candidates
are conservatively quarantined and reported; this is not proof that every
cropped duplicate has been removed. Progress prints every 250 images.


In [ ]:
stage('prepare', '--milk-images', MILK_IMAGES,
      '--isic-ground-truth', ISIC_GROUND_TRUTH, '--isic-metadata', ISIC_METADATA,
      '--isic-images', ISIC_IMAGES, '--workers', PREPROCESS_WORKERS,
      '--pca-dimension', PCA_DIMENSION, '--isic-train-cap', ISIC_TRAIN_CAP,
      '--conditions', *CONDITIONS)


## GPU benchmark before the long run
Measures real frozen-encoder forward throughput and original preprocessing.
Selects a batch/precision profile without updating encoder weights. CPU
preprocessing and disk I/O contribute additional time. The selected profile is
reused on resume; changing source/data/settings requires a new output run.


In [ ]:
stage('benchmark', '--precision', FEATURE_PRECISION)
print(json.dumps(json.loads((RUN/'benchmark.json').read_text()), indent=2))


## Extract frozen representations once
Includes clean, blur, JPEG and dark conditions. Perturbations are applied after
the original deterministic preparation; these are controlled simulations, not
new patients. Cache chunks are resumable. PCA, quality scaling and shared
metadata statistics fit **training cases only**. Cached image arrays are not
included in the final download; reduced features and model state are included.


In [ ]:
stage('extract', '--workers', PREPROCESS_WORKERS)


## Train matched binary controls and the candidate
B0: logistic; B1: fixed fusion MLP; B2: MLP + subtype objective; B3: reliability
fusion; B4: reliability fusion + subtype objective. All receive the same
permitted features and development budget. Three neural head seeds are
averaged equally. Image encoders are never fine-tuned in this first study.
A worse candidate is still recorded; selection never sees PAD outcomes.


In [ ]:
stage('train', '--seeds', *SEEDS, '--epochs', EPOCHS, '--patience', PATIENCE,
      '--batch-size', HEAD_BATCH_SIZE)


## Lock calibration, thresholds and all compared models
Calibration uses the `fusion` development split, with separate calibrators for
available input patterns. The 95% sensitivity setting is a calibration target,
not a promise for test cases. Thresholds are not retuned on holdout or PAD.


In [ ]:
stage('lock', '--target-sensitivity', TARGET_SENSITIVITY)


## Internal checks: same eligible binary cases
Historical MILK validation/holdout have been inspected before and remain
exploratory. Original saved predictions are compared on the new eligible
holdout subset without changing their labels or the archived metrics.


In [ ]:
stage('evaluate', '--bootstrap', BOOTSTRAP_DRAWS)
print((RUN/'development_comparison.md').read_text())


## Open PAD-UFES-20 for external testing only
Requires the intact calibration lock. No external fitting, calibration or
threshold selection occurs. Multiple images are averaged per lesion, and
uncertainty resamples patients. Overlapping patient groups are quarantined
using image identity/pixels independently of model errors. Clinical + metadata
and clinical-only conditions are compared with matched available-input controls.


In [ ]:
stage('external', '--pad-metadata', PAD_METADATA, '--pad-images', PAD_IMAGES,
      '--workers', PREPROCESS_WORKERS, '--bootstrap', BOOTSTRAP_DRAWS)
print((RUN/'external_comparison.md').read_text())
stage('plots')
from IPython.display import display, Image as DisplayImage
for figure in sorted((RUN/'figures').glob('external_*.png')):
    display(DisplayImage(filename=str(figure)))


## Save outputs and inspect the evidence
Download the artifact ZIP. Read `external_results.json`, `external_audit.json`
and `external_predictions.csv` together. They include actual sensitivity and
specificity, per-cancer misses, seed results, calibration, conditions and paired
patient-bootstrap differences. No result is automatically installed into the
app or written over the saved baseline. Do not call a gain established if the
uncertainty or sensitivity guardrails do not support it.


In [ ]:
run_cli('verify')
run_cli('package', '--destination', ARTIFACT_ZIP)
from IPython.display import display, FileLink
display(FileLink(str(ARTIFACT_ZIP)))
